In [1]:
import pandas as pd

imd = pd.read_csv('data/imd_2019.csv', encoding='utf-8-sig')
print(imd.shape)
print(imd.columns.tolist())
print(imd.head(3))

(32844, 6)
['FID', 'LSOA11CD', 'LSOA11NM', 'LAD19CD', 'LAD19NM', 'IMD19']
   FID   LSOA11CD                   LSOA11NM    LAD19CD               LAD19NM  \
0    1  E01000001        City of London 001A  E09000001        City of London   
1    2  E01000104  Barking and Dagenham 014B  E09000002  Barking and Dagenham   
2    3  E01000205                Barnet 035A  E09000003                Barnet   

   IMD19  
0  29199  
1   6002  
2  27397  


In [2]:
import geopandas as gpd

gdf = gpd.read_file('data/lsoa_boundaries.geojson')
print(gdf.shape)
print(gdf.columns.tolist())
print(gdf.head(3))

(34753, 10)
['FID', 'LSOA11CD', 'LSOA11NM', 'LSOA11NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']
   FID   LSOA11CD             LSOA11NM            LSOA11NMW   BNG_E   BNG_N  \
0    1  E01000001  City of London 001A  City of London 001A  532123  181632   
1    2  E01000002  City of London 001B  City of London 001B  532480  181715   
2    3  E01000003  City of London 001C  City of London 001C  532239  182033   

      LONG       LAT                              GlobalID  \
0 -0.09714  51.51816  a758442e-7679-45d0-95a8-ed4c968ecdaa   
1 -0.09197  51.51882  861dbb53-dfaf-4f57-be96-4527e2ec511f   
2 -0.09532  51.52174  9f765b55-2061-484a-862b-fa0325991616   

                                            geometry  
0  POLYGON ((532282.629 181906.496, 532248.25 181...  
1  POLYGON ((532746.814 181786.892, 532248.25 181...  
2  POLYGON ((532293.068 182068.422, 532419.592 18...  


In [3]:
IMD_LSOA_COL = 'LSOA11CD'
IMD_SCORE_COL = 'IMD19'
GEO_LSOA_COL = 'LSOA11CD'

merged = gdf.merge(imd[[IMD_LSOA_COL, IMD_SCORE_COL]], 
                   left_on=GEO_LSOA_COL, 
                   right_on=IMD_LSOA_COL, 
                   how='inner')

print(f"Merged rows: {len(merged)}")
print(merged[[GEO_LSOA_COL, IMD_SCORE_COL]].head())

Merged rows: 32844
    LSOA11CD  IMD19
0  E01000001  29199
1  E01000002  30379
2  E01000003  14915
3  E01000005   8678
4  E01000006  14486


In [4]:
merged = merged.to_crs('EPSG:4326')

merged['centroid'] = merged.geometry.to_crs('EPSG:27700').centroid.to_crs('EPSG:4326')
merged['lat'] = merged['centroid'].y
merged['lon'] = merged['centroid'].x

london = merged[
    (merged['lat'] > 51.28) & (merged['lat'] < 51.70) &
    (merged['lon'] > -0.51) & (merged['lon'] < 0.33)
]

print(f"London LSOAs: {len(london)}")

London LSOAs: 5537


In [5]:
london['decile'] = pd.qcut(london[IMD_SCORE_COL], q=10, labels=False)
sample = london.groupby('decile').apply(
    lambda x: x.sample(200, random_state=42)
).reset_index(drop=True)
sample = sample[[GEO_LSOA_COL, IMD_SCORE_COL, 'lat', 'lon']]

print(f"Total LSOAs sampled: {len(sample)}")
print(sample.head())
sample.to_csv('data/test_lsoas.csv', index=False)
print("Saved!")

Total LSOAs sampled: 2000
    LSOA11CD  IMD19        lat       lon
0  E01000891   4795  51.550666 -0.152359
1  E01000868   4488  51.542270 -0.128073
2  E01002033   4748  51.596099 -0.104196
3  E01004241   5227  51.521003 -0.023303
4  E01001486   3856  51.623352 -0.049548
Saved!
